### Preprocessing

In [6]:
import cv2
import numpy as np

# Try importing rembg with fallback handling
try:
    from rembg import remove
    REMBG_AVAILABLE = True
except ImportError:
    REMBG_AVAILABLE = False
    print("Warning: rembg module not found. Falling back to GrabCut.")

# --------------------------------------------------
# Resize Image
# --------------------------------------------------
def resize_image(image, size=(640, 640)):
    h, w = image.shape[:2]

    target_w, target_h = size


    # calculate scaling ratio
    scale = min(target_w / w, target_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)


    # resize while keeping ratio
    resized = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )


    # create blank canvas
    canvas = np.zeros(
        (target_h, target_w, 3),
        dtype=np.uint8
    )


    # calculate padding
    x_offset = (target_w - new_w)//2
    y_offset = (target_h - new_h)//2


    # place image
    canvas[
        y_offset:y_offset+new_h,
        x_offset:x_offset+new_w
    ] = resized


    return canvas
    # return cv2.resize(image, size, interpolation=cv2.INTER_AREA)


# --------------------------------------------------
# Median Filtering
# --------------------------------------------------
def remove_noise(image):
    return cv2.medianBlur(image, 5)


# --------------------------------------------------
# CLAHE Contrast Enhancement
# --------------------------------------------------
def enhance_contrast(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    l = clahe.apply(l)

    enhanced = cv2.merge((l,a,b))

    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)


# ============================================================
# Background Removal using GrabCut
# ============================================================
def remove_background(image):

    mask = np.zeros(image.shape[:2], np.uint8)

    bgdModel = np.zeros((1,65), np.float64)
    fgdModel = np.zeros((1,65), np.float64)

    height, width = image.shape[:2]

    rect = (
        10,
        10,
        width-20,
        height-20
    )

    cv2.grabCut(
        image,
        mask,
        rect,
        bgdModel,
        fgdModel,
        5,
        cv2.GC_INIT_WITH_RECT
    )

    mask = np.where(
        (mask==2)|(mask==0),
        0,
        1
    ).astype("uint8")

    return image * mask[:, :, np.newaxis], mask

# ============================================================
# Background Removal using rembg (AI model)
# ============================================================
def remove_background_rembg(image):
    if not REMBG_AVAILABLE:
        return remove_background(image)
    
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    output_rgba = remove(rgb_image)
    output_rgb = output_rgba[:, :, :3]
    mask = (output_rgba[:, :, 3] > 0).astype(np.uint8)
    bgr_output = cv2.cvtColor(output_rgb, cv2.COLOR_RGB2BGR)
    return cv2.bitwise_and(bgr_output, bgr_output, mask=mask), mask

# ============================================================
# Morphological Cleaning
# ============================================================
def clean_mask(mask):

    kernel = np.ones((5,5), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    return mask

# ============================================================
# Apply Clean Mask
# ============================================================
def apply_mask(image, mask):

    return cv2.bitwise_and(
        image,
        image,
        mask=mask
    )

# --------------------------------------------------
# Complete Preprocessing Pipeline
# --------------------------------------------------
def preprocess_image(image, use_rembg=True):

    image = resize_image(image)
    image = remove_noise(image)
    image = enhance_contrast(image)
    if use_rembg and REMBG_AVAILABLE:
        image, mask = remove_background_rembg(image)
    else:
        image, mask = remove_background(image)
    mask = clean_mask(mask)
    image = apply_mask(image, mask)
    return image


### Save Cleaned Image To Folder

In [8]:
import os
import glob
import cv2
from tqdm import tqdm

# ==========================
# Dataset paths
# ==========================
source_root = r"../data"
clean_root = r"../cleaned_data"

# Classes
folders = [
    "overripe",
    "fully_ripe",
    "unripe"
]

# Image extensions
extensions = [
    "*.jpg",
    "*.jpeg",
    "*.png"
]

# ==========================
# PROCESS DATASET
# ==========================
for folder in folders:
    print("\nProcessing:", folder)

    source_dir = os.path.join(
        source_root,
        folder
    )

    clean_dir = os.path.join(
        clean_root,
        folder
    )

    # Create cleaned folder
    os.makedirs(
        clean_dir,
        exist_ok=True
    )

    # Get images
    img_paths = []

    for ext in extensions:

        img_paths.extend(
            glob.glob(
                os.path.join(
                    source_dir,
                    ext
                )
            )
        )

    img_paths = sorted(img_paths)

    valid_count = 0
    skipped_count = 0

    # Loop images
    for img_path in tqdm(img_paths):
        if valid_count == 200:
            break
        
        img_name = os.path.basename(
            img_path
        )

        save_path = os.path.join(
            clean_dir,
            img_name
        )

        try:
            # Read image using OpenCV
            image = cv2.imread(
                img_path
            )

            # Check image loaded
            if image is None:

                skipped_count += 1
                continue

            # ==========================
            # PREPROCESS FUNCTION
            # ==========================
            cleaned_image = preprocess_image(
                image
            )

            # Save processed image
            cv2.imwrite(
                save_path,
                cleaned_image
            )

            valid_count += 1

        except Exception as e:
            print(
                "\nFailed:",
                img_name
            )

            print(e)

            skipped_count += 1

    print("Valid images :", valid_count)
    print("Skipped      :", skipped_count)

print("\n====================")
print("Cleaning completed!")
print("====================")


Processing: overripe


 49%|████▉     | 200/408 [06:30<06:46,  1.95s/it]


Valid images : 200
Skipped      : 0

Processing: fully_ripe


 84%|████████▍ | 200/237 [06:05<01:07,  1.83s/it]


Valid images : 200
Skipped      : 0

Processing: unripe


 84%|████████▍ | 200/238 [06:07<01:09,  1.84s/it]

Valid images : 200
Skipped      : 0

Cleaning completed!


### Testing Code

In [ ]:
# 1. Load the image into a variable using imread
original_image = cv2.imread(r'..\data\overripe\image_0_26.jpg')

# 2. Check if the image loaded successfully
if original_image is not None:
    cv2.imshow('Original', original_image)
    processed_image = preprocess_image(original_image)
    cv2.imshow('Processed', processed_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("Error: Could not load image. Check file path.")